In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-rag-eval"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 1. 문서 로드 — kca_report

In [ ]:
PDF_PATHS = [
    PROJECT_ROOT / "data/raw/pdf/kca_report/0417_kca_split.pdf",
]

In [ ]:
from langchain_community.document_loaders import PDFPlumberLoader

all_docs = []
for path in PDF_PATHS:
    loader = PDFPlumberLoader(str(path))
    docs = loader.load()
    for d in docs:
        d.metadata["source"] = path.name
    all_docs.extend(docs)

print("총 문서 수:", len(all_docs))
print(all_docs[0].page_content[:500])

# 2. 문서 split

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
split_docs = text_splitter.split_documents(all_docs)
print(f"청킹 후 문서 수: {len(split_docs)}")

# 3. 임베딩 + 벡터 DB

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
print(f"벡터 수: {vectorstore.index.ntotal}")

# 4. Retriever 설정

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 5. 문서 내용 확인 (질문 설계용)

> 아래 셀을 실행해서 문서 내용을 파악한 뒤, 섹션 6의 questions/ground_truths를 실제 내용에 맞게 수정하세요.

In [ ]:
# 문서 내용 샘플 확인
for i, doc in enumerate(all_docs[:5]):
    print(f"\n--- 페이지 {i+1} ---")
    print(doc.page_content[:300])

# 6. 질문 → context → 답변 생성

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=300)

def run_rag(q):
    docs = retriever.invoke(q)
    context_texts = [doc.page_content[:400] for doc in docs]
    answer = llm.invoke(
        f"""질문: {q}

아래 문서에 있는 내용만 사용해서 핵심 답변을 1~2문장으로 작성하세요.
문서에 없는 내용은 절대 추가하지 마세요.

문서:
{context_texts}"""
    ).content
    sources = [doc.metadata.get("source", "출처 없음") for doc in docs]
    return answer, context_texts, sources

def target(inputs: dict):
    q = inputs["question"]
    answer, contexts, sources = run_rag(q)
    return {"answer": answer, "contexts": contexts}

In [ ]:
# 단건 테스트 — 문서 내용 확인 후 질문을 바꿔도 됩니다
q = "이 보고서에서 다루는 주요 소비자 이슈는 무엇인가?"
answer, contexts, sources = run_rag(q)
print("답변:", answer)
print("출처:", sources)

# 7. LangSmith Dataset 생성

> ⚠️ 위 단건 테스트로 문서 내용을 파악한 뒤, questions와 ground_truths를 실제 내용에 맞게 수정하세요.

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "catcher-rag-kca-eval"

# ⚠️ 문서 내용 확인 후 수정하세요
questions = [
    "이 보고서에서 다루는 주요 소비자 이슈는 무엇인가?",
    "소비자 피해가 가장 많은 분야는 어디인가?",
    "소비자 불만 처리 방법은 어떻게 되는가?",
    "온라인 거래 관련 소비자 주의사항은 무엇인가?",
    "보고서에서 권고하는 소비자 보호 방안은?",
]

# ⚠️ 실제 문서 내용으로 수정 필요
ground_truths = [
    "(문서 내용 확인 후 작성)",
    "(문서 내용 확인 후 작성)",
    "(문서 내용 확인 후 작성)",
    "(문서 내용 확인 후 작성)",
    "(문서 내용 확인 후 작성)",
]

existing = [d for d in client.list_datasets() if d.name == dataset_name]
if existing:
    dataset = existing[0]
    print(f"기존 dataset 사용: {dataset.name}")
else:
    dataset = client.create_dataset(dataset_name=dataset_name, description="kca_report RAG 평가")
    for q, gt in zip(questions, ground_truths):
        client.create_example(
            inputs={"question": q},
            outputs={"ground_truth": gt},
            dataset_id=dataset.id
        )
    print(f"새 dataset 생성: {dataset.name} ({len(questions)}개)")

# 8. Evaluator 정의

In [ ]:
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def correctness_evaluator(run, example):
    answer = run.outputs["answer"]
    ground_truth = example.outputs["ground_truth"]
    prompt = f"""다음 답변이 정답과 얼마나 일치하는지 0~1 점수로 평가해줘.
정답: {ground_truth}
답변: {answer}
숫자 하나만 출력해."""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "correctness", "score": float(score)}

def faithfulness_evaluator(run, example):
    answer = run.outputs["answer"]
    contexts = run.outputs["contexts"]
    prompt = f"""아래 답변이 문서에 있는 내용만 사용했는지 0~1로 평가해줘.
숫자 하나만 출력해.
문서: {contexts}
답변: {answer}"""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "faithfulness", "score": float(score)}

print("evaluator 2개 정의 완료")

# 9. evaluate() 실행 → LangSmith 반영

In [ ]:
from langsmith.evaluation import evaluate

evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness_evaluator, faithfulness_evaluator],
    experiment_prefix="kca-rag-v1"
)